# E011 — DiffQRCoder vs QRBTF public, 4 prompts, puis SR-MPGD

Ce notebook compare uniquement deux familles sur le même payload, la même matrice QR, les mêmes quatre prompts et les mêmes seeds.

- **DiffQRCoder-paper** : Cetus-Mix Whalefall + QR Code Monster v2, Stage 1, initialisation bruitée depuis le Stage 1, cible QArt-compatible, puis SRPG.
- **QRBTF-public-reproduction** : reproduction locale du workflow public ControlNet, avec QR Code Monster v2 principal et Brightness ControlNet auxiliaire.

Le backend IA exact de QRBTF n'est pas open source et ne fournit pas ses latents intermédiaires. Cette seconde branche n'est donc jamais appelée « QRBTF officiel ». Chaque méthode est ensuite testée sans et avec le même post-traitement SR-MPGD.

## Matrice expérimentale

| Méthode | Variante | Sortie évaluée |
|---|---|---|
| DiffQRCoder-paper | base | Stage 2 SRPG |
| DiffQRCoder-paper | + SR-MPGD | même image puis optimisation latente |
| QRBTF-public-reproduction | base | text2img double ControlNet |
| QRBTF-public-reproduction | + SR-MPGD | même image puis optimisation latente |

Chaque débruitage sauvegarde **tous ses pas** en JPEG et fabrique un GIF. Les scores sont : SSR logiciel exact, SSR original, MER, CLIP-aesthetic, CLIPScore et temps. Le SSR physique reste un protocole séparé à remplir sur téléphone.

In [ ]:
from __future__ import annotations

import csv
import gc
import hashlib
import importlib.metadata
import json
import shutil
import sys
import time
from dataclasses import asdict
from datetime import UTC, datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import qrcode
import torch
from diffusers import (
    ControlNetModel,
    DDIMScheduler,
    DPMSolverMultistepScheduler,
    StableDiffusionControlNetPipeline,
)
from IPython.display import Markdown, clear_output, display
from PIL import Image
from qrcode.exceptions import DataOverflowError

UPSTREAM_ROOT = Path("/opt/DiffQRCoder")
DIFFQRCODER_COMMIT = "e24ea73ee2e13c7e6e87cb422e8b11784e70ae00"
sys.path.insert(0, str(UPSTREAM_ROOT))
from diffqrcoder import DiffQRCoderPipeline  # noqa: E402
from diffqrcoder.image_processor import crop_padding, image_binarize  # noqa: E402
from diffqrcoder.losses.perceptual_loss import PerceptualLoss  # noqa: E402
from diffqrcoder.srpg import ScanningRobustPerceptualGuidance  # noqa: E402

from prooftag_qr.qr import QRBlueprint, functional_pattern_mask, module_error_rate  # noqa: E402
from prooftag_qr.quality import image_quality_metrics  # noqa: E402
from prooftag_qr.quality_scoring import CLIPQualityScorer  # noqa: E402
from prooftag_qr.validation import QRValidator, summarize_validation_records  # noqa: E402


def differentiable_perceptual_forward(self, x, y):
    losses = [
        torch.nn.functional.mse_loss(fx, fy)
        for fx, fy in zip(self.extractor(x), self.extractor(y), strict=True)
    ]
    return torch.stack(losses).mean()


PerceptualLoss.forward = differentiable_perceptual_forward

assert torch.cuda.is_available(), "Exécuter ce notebook dans le pod GPU."
assert importlib.metadata.version("diffusers") == "0.32.2"
print(torch.__version__, torch.cuda.get_device_name(0), importlib.metadata.version("diffusers"))

## 1. Paramètres figés et quatre niveaux de complexité

Modifier le payload seulement s'il reste compatible avec QR version 3/M. Les quatre prompts sont volontairement simple, moyen, détaillé et complexe. Chaque prompt possède une seed fixe utilisée dans les deux branches.

In [ ]:
EXPERIMENT_NAME = "e011-diffqrcoder-vs-qrbtf-public-v1"
RESUME_RUN_NAME = ""  # renseigner un dossier existant après une interruption
PAYLOAD = "Thanks reviewer!"  # remplacer ensuite par une URL Prooftag courte
NEGATIVE_PROMPT = "easynegative, low quality, blurry, text, watermark, distorted geometry"
PROMPTS = [
    {
        "id": "p1_simple",
        "seed": 101,
        "text": "A purple snake, centered composition, high contrast.",
    },
    {
        "id": "p2_medium",
        "seed": 202,
        "text": "White and pale blue botanical flowers, dark leaves, elegant flat illustration, high contrast.",
    },
    {
        "id": "p3_detailed",
        "seed": 303,
        "text": "Winter wonderland, fresh snowfall, evergreen trees, cozy log cabin, aurora borealis, cinematic night.",
    },
    {
        "id": "p4_complex",
        "seed": 404,
        "text": "An ornate Japanese temple above a river, cherry blossoms, stone lanterns, distant mountains, dramatic sunset, intricate watercolor.",
    },
]

BASE_MODEL = "https://huggingface.co/fp16-guy/Cetus-Mix_Whalefall_fp16_cleaned/blob/main/cetusMix_Whalefall2_fp16.safetensors"
MONSTER_MODEL = "monster-labs/control_v1p_sd15_qrcode_monster"
MONSTER_SUBFOLDER = "v2"
BRIGHTNESS_MODEL = "latentcat/control_v1p_sd15_brightness"

QR_VERSION, QR_MASK, MODULE_SIZE, BORDER = 3, 4, 16, 4
PADDING_PX = BORDER * MODULE_SIZE
IMAGE_SIZE = (29 + 2 * BORDER) * MODULE_SIZE  # 592 px, condition native conseillée par Monster v2
STEPS, CFG, CONTROL_SCALE = 40, 7.5, 1.35
SRG, PG = 500, 3
SRMPGD_ITERATIONS, SRMPGD_LR = 20, 0.1
QRBTF_MONSTER_SCALE, QRBTF_BRIGHTNESS_SCALE = 1.0, 0.25
QRBTF_START, QRBTF_END = [0.0, 0.4], [1.0, 0.8]

RUN_NAME = RESUME_RUN_NAME or f"{datetime.now(UTC).strftime('%Y%m%dT%H%M%SZ')}-{EXPERIMENT_NAME}"
RUN_DIR = Path("/data/notebook-runs") / RUN_NAME
if RESUME_RUN_NAME and not RUN_DIR.is_dir():
    raise FileNotFoundError(f"Run à reprendre introuvable : {RUN_DIR}")
RUN_DIR.mkdir(parents=True, exist_ok=bool(RESUME_RUN_NAME))
(RUN_DIR / "frames").mkdir(exist_ok=True)
print("RUN_DIR =", RUN_DIR)
print("DiffQRCoder :", BASE_MODEL, "+", f"{MONSTER_MODEL}/v2")
print("QRBTF public:", BASE_MODEL, "+", f"{MONSTER_MODEL}/v2", "+", BRIGHTNESS_MODEL)

In [ ]:
qr = qrcode.QRCode(
    version=QR_VERSION,
    error_correction=qrcode.constants.ERROR_CORRECT_M,
    box_size=MODULE_SIZE,
    border=BORDER,
    mask_pattern=QR_MASK,
)
qr.add_data(PAYLOAD)
try:
    qr.make(fit=False)
except DataOverflowError as exc:
    raise ValueError("Payload trop long pour QR v3/M : utiliser une URL courte.") from exc
qr_image = qr.make_image(fill_color="black", back_color="white").convert("RGB")
matrix = np.asarray(qr.get_matrix(), dtype=np.uint8)
blueprint = QRBlueprint(qr_image, matrix, QR_VERSION, BORDER)
assert qr_image.size == (IMAGE_SIZE, IMAGE_SIZE) == (592, 592)
qr_image.save(RUN_DIR / "00_shared_qr.png")


def gray_quiet_condition(source):
    array = np.asarray(source).copy()
    p = PADDING_PX
    array[:p, :], array[-p:, :], array[:, :p], array[:, -p:] = 128, 128, 128, 128
    return Image.fromarray(array)


monster_condition = gray_quiet_condition(qr_image)
brightness_condition = monster_condition.copy()
monster_condition.save(RUN_DIR / "00_qrbtf_monster_condition.png")
display(
    Markdown(
        f"**QR partagé :** version {QR_VERSION}, M (15 %), masque {QR_MASK}, {matrix.shape[0]}×{matrix.shape[1]} modules, {IMAGE_SIZE}px"
    )
)
display(qr_image.resize((296, 296)), monster_condition.resize((296, 296)))

## 2. Instrumentation commune

Les callbacks sauvegardent chaque estimation latente décodée. L'affichage live est limité à un pas sur cinq, mais les 40 fichiers restent présents. Le SSR logiciel est le nombre de payloads exacts divisé par tous les couples décodeur × dégradation. `SSR_original` ne retient que l'image non dégradée.

In [ ]:
def cuda_memory():
    free, total = torch.cuda.mem_get_info()
    return {
        "allocated_gib": torch.cuda.memory_allocated() / 2**30,
        "free_gib": free / 2**30,
        "total_gib": total / 2**30,
    }


def release_pipeline(name):
    value = globals().pop(name, None)
    if value is not None:
        try:
            value.to("cpu")
        except Exception:
            pass
        del value
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print("VRAM:", cuda_memory())


for stale_name in ["pipe", "controlnet", "diff_pipe", "qrbtf_pipe"]:
    release_pipeline(stale_name)
if torch.cuda.mem_get_info()[0] / 2**30 < 15:
    raise RuntimeError(
        "Moins de 15 Gio GPU libres. Redémarrer le kernel et arrêter les autres workloads."
    )


def decode_latents(pipe, latents):
    with torch.no_grad():
        decoded = pipe.vae.decode(
            latents.detach() / pipe.vae.config.scaling_factor, return_dict=False
        )[0]
        return pipe.image_processor.postprocess(decoded, output_type="pil")[0]


def frame_callback(pipe, prompt_id, phase, total_steps):
    folder = RUN_DIR / "frames" / prompt_id / phase
    folder.mkdir(parents=True, exist_ok=True)
    trace, started = [], time.perf_counter()

    def callback(pipeline, step, timestep, kwargs):
        image = decode_latents(pipeline, kwargs["latents"])
        row = {
            "step": int(step),
            "timestep": int(timestep),
            "elapsed_s": time.perf_counter() - started,
            "mer": module_error_rate(image, blueprint),
        }
        trace.append(row)
        image.save(folder / f"{step:03d}.jpg", quality=88)
        if step % 5 == 0 or step == total_steps - 1:
            clear_output(wait=True)
            display(
                Markdown(
                    f"**{prompt_id} / {phase} — {step + 1}/{total_steps}, MER {row['mer']:.2%}**"
                )
            )
            display(image.resize((420, 420)))
        return kwargs

    return callback, trace, folder


def make_gif(folder, output):
    paths = sorted(folder.glob("*.jpg"))
    if not paths:
        return None
    frames = [Image.open(path).convert("RGB").resize((296, 296)) for path in paths]
    frames[0].save(
        output, save_all=True, append_images=frames[1:], duration=180, loop=0, optimize=True
    )
    return output


validator = QRValidator()
quality_scorer = CLIPQualityScorer(Path("/cache"), device="cpu")
RESULTS_PATH = RUN_DIR / "results.jsonl"
rows = []
if RESULTS_PATH.exists():
    rows = [
        json.loads(line)
        for line in RESULTS_PATH.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    print(f"Reprise : {len(rows)} résultats déjà chargés")


def completed(prompt_id, method, variant):
    return any(
        row["prompt_id"] == prompt_id and row["method"] == method and row["variant"] == variant
        for row in rows
    )


def reset_incomplete(prompt_id, method):
    matching = [row for row in rows if row["prompt_id"] == prompt_id and row["method"] == method]
    if not matching:
        return
    rows[:] = [
        row for row in rows if not (row["prompt_id"] == prompt_id and row["method"] == method)
    ]
    RESULTS_PATH.write_text("".join(json.dumps(row) + "\n" for row in rows), encoding="utf-8")
    print(f"Reprise : couple incomplet recalculé — {prompt_id}/{method}")


def evaluate(prompt_case, method, variant, image, generation_s, incremental_s, parameters):
    records = validator.validate(image, PAYLOAD)
    exact = [record for record in records if record.exact_payload_match]
    originals = [record for record in records if record.scenario == "original"]
    summary = summarize_validation_records(records)
    quality = quality_scorer.score(image, prompt_case["text"])
    row = {
        "prompt_id": prompt_case["id"],
        "prompt": prompt_case["text"],
        "seed": prompt_case["seed"],
        "method": method,
        "variant": variant,
        "passed": len(exact),
        "total": len(records),
        "software_ssr": len(exact) / len(records),
        "original_ssr": sum(r.exact_payload_match for r in originals) / len(originals),
        "strict_all": len(exact) == len(records),
        "worst_decoder_ssr": summary["worst_decoder_pass_rate"],
        "mer": module_error_rate(image, blueprint),
        "clip_aesthetic": quality.clip_aesthetic,
        "clip_score": quality.clip_score,
        "clip_similarity": quality.clip_similarity,
        "generation_s": generation_s,
        "incremental_s": incremental_s,
        "parameters": parameters,
        **image_quality_metrics(image),
    }
    rows.append(row)
    with RESULTS_PATH.open("a", encoding="utf-8") as stream:
        stream.write(json.dumps(row) + "\n")
    validation_path = RUN_DIR / prompt_case["id"] / method / f"{variant}-validations.json"
    validation_path.parent.mkdir(parents=True, exist_ok=True)
    validation_path.write_text(json.dumps([asdict(r) for r in records], indent=2), encoding="utf-8")
    print(
        f"{method}/{variant}: SSR={row['passed']}/{row['total']} ({row['software_ssr']:.1%}), aes={row['clip_aesthetic']:.3f}, CLIPScore={row['clip_score']:.3f}, {generation_s:.1f}s"
    )
    return row

In [ ]:
@torch.no_grad()
def latent_from_image(pipe, image):
    tensor = pipe.image_processor.preprocess(image, height=IMAGE_SIZE, width=IMAGE_SIZE).to(
        "cuda", dtype=torch.float16
    )
    return pipe.vae.encode(tensor).latent_dist.mode() * pipe.vae.config.scaling_factor


def apply_srmpgd(pipe, image, prompt_case, method, target_image):
    phase = f"{method}_srmpgd"
    folder = RUN_DIR / "frames" / prompt_case["id"] / phase
    folder.mkdir(parents=True, exist_ok=True)
    ref_norm = pipe.image_processor.preprocess(image, height=IMAGE_SIZE, width=IMAGE_SIZE).to(
        "cuda", dtype=torch.float16
    )
    ref01 = ref_norm / 2 + 0.5
    target = (
        torch.from_numpy(np.asarray(target_image).copy())
        .permute(2, 0, 1)[None]
        .to("cuda", dtype=torch.float16)
        / 255
    )
    latents = latent_from_image(pipe, image).detach().requires_grad_(True)
    guidance = ScanningRobustPerceptualGuidance(MODULE_SIZE, SRG, PG).to(
        "cuda", dtype=torch.float16
    )
    optimizer = torch.optim.SGD([latents], lr=SRMPGD_LR)
    trace, started = [], time.perf_counter()
    for step in range(SRMPGD_ITERATIONS):
        optimizer.zero_grad()
        decoded_norm = pipe.vae.decode(latents / pipe.vae.config.scaling_factor, return_dict=False)[
            0
        ]
        decoded01 = pipe.image_processor.denormalize(decoded_norm)
        loss = guidance.compute_loss(
            crop_padding(decoded01, PADDING_PX),
            crop_padding(image_binarize(target), PADDING_PX),
            crop_padding(ref01, PADDING_PX),
        )
        loss.backward()
        optimizer.step()
        preview = decode_latents(pipe, latents)
        row = {
            "step": step,
            "loss": float(loss.detach()),
            "mer": module_error_rate(preview, blueprint),
            "elapsed_s": time.perf_counter() - started,
        }
        trace.append(row)
        preview.save(folder / f"{step:03d}.jpg", quality=88)
        clear_output(wait=True)
        display(
            Markdown(
                f"**{prompt_case['id']} / {phase} — {step + 1}/{SRMPGD_ITERATIONS}, MER {row['mer']:.2%}**"
            )
        )
        display(preview.resize((420, 420)))
    result = decode_latents(pipe, latents)
    duration = time.perf_counter() - started
    del guidance, optimizer, latents
    gc.collect()
    torch.cuda.empty_cache()
    (RUN_DIR / prompt_case["id"] / method / f"{method}-srmpgd-trace.json").write_text(
        json.dumps(trace, indent=2), encoding="utf-8"
    )
    make_gif(folder, RUN_DIR / prompt_case["id"] / method / f"{method}-srmpgd.gif")
    return result, duration

## 3. Branche DiffQRCoder-paper

E010 appelait le dépôt public tel quel, dont le Stage 2 repart actuellement d'un bruit aléatoire. Ici, l'initialisation suit l'algorithme du papier : encodage du Stage 1 puis ajout de bruit. Le dépôt ne fournit pas l'implémentation Reed–Solomon de QArt ; la cible `qart_proxy` conserve donc exactement la matrice et rapproche uniquement les centres des modules de la luminance du Stage 1. Elle sert de **condition**, jamais de collage final.

In [ ]:
def load_diffqrcoder():
    release_pipeline("diff_pipe")
    control = ControlNetModel.from_pretrained(
        MONSTER_MODEL,
        subfolder=MONSTER_SUBFOLDER,
        torch_dtype=torch.float16,
        cache_dir="/cache/huggingface",
    )
    pipe = DiffQRCoderPipeline.from_single_file(
        BASE_MODEL,
        controlnet=control,
        torch_dtype=torch.float16,
        cache_dir="/cache/huggingface",
        safety_checker=None,
        use_safetensors=True,
    )
    pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
    pipe = pipe.to("cuda")
    pipe.enable_attention_slicing("max")
    pipe.enable_vae_slicing()
    for module in [pipe.unet, pipe.controlnet, pipe.vae, pipe.text_encoder]:
        module.requires_grad_(False).eval()
    pipe.unet.enable_gradient_checkpointing()
    pipe.controlnet.enable_gradient_checkpointing()
    return pipe


def qart_proxy(stage1):
    source = np.asarray(stage1.resize((IMAGE_SIZE, IMAGE_SIZE))).astype(np.float32)
    function = functional_pattern_mask(blueprint)
    count = matrix.shape[0]
    for row in range(count):
        for col in range(count):
            y0, y1 = row * MODULE_SIZE, (row + 1) * MODULE_SIZE
            x0, x1 = col * MODULE_SIZE, (col + 1) * MODULE_SIZE
            fraction = 0.72 if function[row, col] else 0.38
            margin = max(1, round(MODULE_SIZE * (1 - fraction) / 2))
            target = 0 if matrix[row, col] else 255
            source[y0 + margin : y1 - margin, x0 + margin : x1 - margin] = (
                0.15 * source[y0 + margin : y1 - margin, x0 + margin : x1 - margin] + 0.85 * target
            )
    source[:PADDING_PX, :] = 255
    source[-PADDING_PX:, :] = 255
    source[:, :PADDING_PX] = 255
    source[:, -PADDING_PX:] = 255
    return Image.fromarray(np.clip(source, 0, 255).astype(np.uint8))


@torch.no_grad()
def paper_stage2_latents(pipe, stage1_tensor, seed):
    normalized = stage1_tensor.to("cuda", dtype=torch.float16) * 2 - 1
    encoded = pipe.vae.encode(normalized).latent_dist.mode() * pipe.vae.config.scaling_factor
    generator = torch.Generator(device="cuda").manual_seed(seed)
    noise = torch.randn(encoded.shape, generator=generator, device="cuda", dtype=encoded.dtype)
    pipe.scheduler.set_timesteps(STEPS, device="cuda")
    return pipe.scheduler.add_noise(encoded, noise, pipe.scheduler.timesteps[:1])


@torch.no_grad()
def run_diffqrcoder_case(pipe, case):
    output_dir = RUN_DIR / case["id"] / "diffqrcoder_paper"
    output_dir.mkdir(parents=True, exist_ok=True)
    stage1_started = time.perf_counter()
    cb1, trace1, frames1 = frame_callback(pipe, case["id"], "diff_stage1", STEPS)
    generator = torch.Generator(device="cuda").manual_seed(case["seed"])
    stage1 = pipe._run_stage1(
        prompt=case["text"],
        qrcode=monster_condition,
        height=IMAGE_SIZE,
        width=IMAGE_SIZE,
        negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=STEPS,
        guidance_scale=CFG,
        generator=generator,
        controlnet_conditioning_scale=CONTROL_SCALE,
        output_type="pt",
        callback_on_step_end=cb1,
        callback_on_step_end_tensor_inputs=["latents"],
    )
    stage1_s = time.perf_counter() - stage1_started
    stage1_tensor = stage1.images.detach()
    stage1_image = pipe.image_processor.numpy_to_pil(
        pipe.image_processor.pt_to_numpy(stage1_tensor)
    )[0]
    stage1_image.save(output_dir / "stage1.png")
    make_gif(frames1, output_dir / "stage1-diffusion.gif")
    target = qart_proxy(stage1_image)
    target.save(output_dir / "qart-proxy-condition.png")
    initial_latents = paper_stage2_latents(pipe, stage1_tensor, case["seed"] + 10000)
    cb2, trace2, frames2 = frame_callback(pipe, case["id"], "diff_stage2_srpg", STEPS)
    stage2_started = time.perf_counter()
    result = pipe._run_stage2(
        prompt=case["text"],
        qrcode=target,
        qrcode_module_size=MODULE_SIZE,
        qrcode_padding=PADDING_PX,
        ref_image=stage1_tensor,
        height=IMAGE_SIZE,
        width=IMAGE_SIZE,
        negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=STEPS,
        guidance_scale=CFG,
        generator=torch.Generator(device="cuda").manual_seed(case["seed"] + 10000),
        latents=initial_latents,
        controlnet_conditioning_scale=CONTROL_SCALE,
        scanning_robust_guidance_scale=SRG,
        perceptual_guidance_scale=PG,
        callback_on_step_end=cb2,
        callback_on_step_end_tensor_inputs=["latents"],
        output_type="pil",
    )
    stage2_s = time.perf_counter() - stage2_started
    image = result.images[0]
    duration = stage1_s + stage2_s
    image.save(output_dir / "base.png")
    make_gif(frames2, output_dir / "stage2-diffusion.gif")
    (output_dir / "diffusion-traces.json").write_text(
        json.dumps({"stage1": trace1, "stage2": trace2}, indent=2), encoding="utf-8"
    )
    return (
        image,
        target,
        duration,
        {"stage1_s": stage1_s, "stage2_s": stage2_s, "total_s": duration},
    )

In [ ]:
diff_pipe = load_diffqrcoder()
for case in PROMPTS:
    if completed(case["id"], "diffqrcoder_paper", "base") and completed(
        case["id"], "diffqrcoder_paper", "with_srmpgd"
    ):
        print("Déjà terminé :", case["id"], "diffqrcoder_paper")
        continue
    reset_incomplete(case["id"], "diffqrcoder_paper")
    base, target, base_s, phase_times = run_diffqrcoder_case(diff_pipe, case)
    params = {
        "base_model": BASE_MODEL,
        "controlnet": f"{MONSTER_MODEL}/v2",
        "scheduler": "DDIM",
        "steps_stage1": STEPS,
        "steps_stage2": STEPS,
        "cfg": CFG,
        "control_scale": CONTROL_SCALE,
        "srg": SRG,
        "pg": PG,
        "stage2_init": "noisy_stage1_encoding",
        "qart": "documented matrix-preserving proxy",
        "phase_times_s": phase_times,
    }
    evaluate(case, "diffqrcoder_paper", "base", base, base_s, base_s, params)
    refined, mpgd_s = apply_srmpgd(diff_pipe, base, case, "diffqrcoder_paper", target)
    refined.save(RUN_DIR / case["id"] / "diffqrcoder_paper" / "with_srmpgd.png")
    evaluate(
        case,
        "diffqrcoder_paper",
        "with_srmpgd",
        refined,
        base_s + mpgd_s,
        mpgd_s,
        {**params, "srmpgd_iterations": SRMPGD_ITERATIONS, "srmpgd_lr": SRMPGD_LR},
    )
release_pipeline("diff_pipe")

## 4. Branche QRBTF-public-reproduction

La documentation publique décrit Stable Diffusion + ControlNet mais pas le checkpoint propriétaire de QRBTF. Pour isoler la méthode, cette branche conserve la même fondation Cetus-Mix. Elle applique QR Monster v2 à 1,0 pendant toute la diffusion et le Brightness ControlNet à 0,25 entre 40 % et 80 %, combinaison publiée pour améliorer la reconnaissance sans pousser Monster à 1,5. Scheduler : DPM++ SDE Karras.

In [ ]:
def load_qrbtf_public():
    release_pipeline("qrbtf_pipe")
    monster = ControlNetModel.from_pretrained(
        MONSTER_MODEL,
        subfolder=MONSTER_SUBFOLDER,
        torch_dtype=torch.float16,
        cache_dir="/cache/huggingface",
    )
    brightness = ControlNetModel.from_pretrained(
        BRIGHTNESS_MODEL, torch_dtype=torch.float16, cache_dir="/cache/huggingface"
    )
    pipe = StableDiffusionControlNetPipeline.from_single_file(
        BASE_MODEL,
        controlnet=[monster, brightness],
        torch_dtype=torch.float16,
        cache_dir="/cache/huggingface",
        safety_checker=None,
        use_safetensors=True,
    )
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(
        pipe.scheduler.config, algorithm_type="sde-dpmsolver++", use_karras_sigmas=True
    )
    pipe = pipe.to("cuda")
    pipe.enable_attention_slicing("max")
    pipe.enable_vae_slicing()
    for module in [pipe.unet, pipe.controlnet, pipe.vae, pipe.text_encoder]:
        module.requires_grad_(False).eval()
    return pipe


@torch.no_grad()
def run_qrbtf_case(pipe, case):
    output_dir = RUN_DIR / case["id"] / "qrbtf_public_reproduction"
    output_dir.mkdir(parents=True, exist_ok=True)
    callback, trace, frames = frame_callback(pipe, case["id"], "qrbtf_public", STEPS)
    started = time.perf_counter()
    result = pipe(
        prompt=case["text"],
        negative_prompt=NEGATIVE_PROMPT,
        image=[monster_condition, brightness_condition],
        height=IMAGE_SIZE,
        width=IMAGE_SIZE,
        num_inference_steps=STEPS,
        guidance_scale=CFG,
        generator=torch.Generator(device="cuda").manual_seed(case["seed"]),
        controlnet_conditioning_scale=[QRBTF_MONSTER_SCALE, QRBTF_BRIGHTNESS_SCALE],
        control_guidance_start=QRBTF_START,
        control_guidance_end=QRBTF_END,
        callback_on_step_end=callback,
        callback_on_step_end_tensor_inputs=["latents"],
    )
    duration = time.perf_counter() - started
    image = result.images[0]
    image.save(output_dir / "base.png")
    make_gif(frames, output_dir / "diffusion.gif")
    (output_dir / "diffusion-trace.json").write_text(json.dumps(trace, indent=2), encoding="utf-8")
    return image, duration


qrbtf_pipe = load_qrbtf_public()
for case in PROMPTS:
    if completed(case["id"], "qrbtf_public_reproduction", "base") and completed(
        case["id"], "qrbtf_public_reproduction", "with_srmpgd"
    ):
        print("Déjà terminé :", case["id"], "qrbtf_public_reproduction")
        continue
    reset_incomplete(case["id"], "qrbtf_public_reproduction")
    base, base_s = run_qrbtf_case(qrbtf_pipe, case)
    params = {
        "label": "QRBTF public reproduction, not proprietary QRBTF",
        "base_model": BASE_MODEL,
        "controlnets": [f"{MONSTER_MODEL}/v2", BRIGHTNESS_MODEL],
        "scales": [QRBTF_MONSTER_SCALE, QRBTF_BRIGHTNESS_SCALE],
        "starts": QRBTF_START,
        "ends": QRBTF_END,
        "scheduler": "DPM++ SDE Karras",
        "steps": STEPS,
        "cfg": CFG,
    }
    evaluate(case, "qrbtf_public_reproduction", "base", base, base_s, base_s, params)
    refined, mpgd_s = apply_srmpgd(qrbtf_pipe, base, case, "qrbtf_public_reproduction", qr_image)
    refined.save(RUN_DIR / case["id"] / "qrbtf_public_reproduction" / "with_srmpgd.png")
    evaluate(
        case,
        "qrbtf_public_reproduction",
        "with_srmpgd",
        refined,
        base_s + mpgd_s,
        mpgd_s,
        {**params, "srmpgd_iterations": SRMPGD_ITERATIONS, "srmpgd_lr": SRMPGD_LR},
    )
release_pipeline("qrbtf_pipe")

## 5. Comparaison finale

Les résultats sont d'abord groupés par prompt pour éviter qu'une moyenne cache un échec. L'agrégat donne ensuite SSR moyen, pire SSR, prompts stricts, esthétique et temps. Aucun résultat physique n'est inventé.

In [ ]:
assert len(rows) == len(PROMPTS) * 4, f"Campagne incomplète : {len(rows)}/16 résultats"
columns = [
    "prompt_id",
    "method",
    "variant",
    "passed",
    "total",
    "software_ssr",
    "original_ssr",
    "mer",
    "clip_aesthetic",
    "clip_score",
    "generation_s",
]
with (RUN_DIR / "comparison.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=columns)
    writer.writeheader()
    writer.writerows([{k: r[k] for k in columns} for r in rows])

keys = [
    ("diffqrcoder_paper", "base"),
    ("diffqrcoder_paper", "with_srmpgd"),
    ("qrbtf_public_reproduction", "base"),
    ("qrbtf_public_reproduction", "with_srmpgd"),
]
aggregates = []
for method, variant in keys:
    group = [r for r in rows if r["method"] == method and r["variant"] == variant]
    aggregates.append(
        {
            "method": method,
            "variant": variant,
            "mean_ssr": float(np.mean([r["software_ssr"] for r in group])),
            "worst_ssr": min(r["software_ssr"] for r in group),
            "strict_prompts": sum(r["strict_all"] for r in group),
            "mean_aesthetic": float(np.mean([r["clip_aesthetic"] for r in group])),
            "mean_clipscore": float(np.mean([r["clip_score"] for r in group])),
            "mean_generation_s": float(np.mean([r["generation_s"] for r in group])),
        }
    )
(RUN_DIR / "aggregates.json").write_text(json.dumps(aggregates, indent=2), encoding="utf-8")
for item in aggregates:
    print(item)

fig, axes = plt.subplots(len(PROMPTS), 4, figsize=(16, 16))
for row_index, case in enumerate(PROMPTS):
    for col_index, (method, variant) in enumerate(keys):
        result = next(
            r
            for r in rows
            if r["prompt_id"] == case["id"] and r["method"] == method and r["variant"] == variant
        )
        filename = "base.png" if variant == "base" else "with_srmpgd.png"
        image = Image.open(RUN_DIR / case["id"] / method / filename)
        axes[row_index, col_index].imshow(image)
        axes[row_index, col_index].axis("off")
        axes[row_index, col_index].set_title(
            f"{case['id']}\n{method.replace('_public_reproduction', '')} / {variant}\nSSR {result['passed']}/{result['total']} | aes {result['clip_aesthetic']:.2f}",
            fontsize=9,
        )
fig.tight_layout()
fig.savefig(RUN_DIR / "comparison-4x4.png", dpi=150, bbox_inches="tight")
display(fig)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()
labels = [f"{m.replace('_public_reproduction', '')}\n{v}" for m, v in keys]
axes[0].bar(labels, [a["mean_ssr"] for a in aggregates])
axes[0].set_title("SSR logiciel moyen")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=20)
axes[1].bar(labels, [a["mean_aesthetic"] for a in aggregates])
axes[1].set_title("CLIP-aesthetic moyen")
axes[1].tick_params(axis="x", rotation=20)
axes[2].bar(labels, [a["mean_clipscore"] for a in aggregates])
axes[2].set_title("CLIPScore moyen")
axes[2].tick_params(axis="x", rotation=20)
axes[3].bar(labels, [a["mean_generation_s"] for a in aggregates])
axes[3].set_title("Temps moyen, secondes")
axes[3].tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(RUN_DIR / "aggregate-metrics.png", dpi=150, bbox_inches="tight")
display(fig)

In [ ]:
manifest = {
    "experiment": EXPERIMENT_NAME,
    "created_at": datetime.now(UTC).isoformat(),
    "payload_sha256": hashlib.sha256(PAYLOAD.encode()).hexdigest(),
    "prompts": PROMPTS,
    "shared_qr": {
        "version": QR_VERSION,
        "ecc": "M",
        "mask": QR_MASK,
        "module_size": MODULE_SIZE,
        "image_size": IMAGE_SIZE,
    },
    "diffqrcoder": {
        "commit": DIFFQRCODER_COMMIT,
        "base": BASE_MODEL,
        "controlnet": f"{MONSTER_MODEL}/v2",
        "stage2_initialization": "paper noisy Stage-1 encoding",
        "qart_limitation": "matrix-preserving visual proxy; Reed-Solomon QArt code absent upstream",
    },
    "qrbtf": {
        "label": "public local reproduction, not proprietary backend",
        "base_for_fairness": BASE_MODEL,
        "controlnets": [f"{MONSTER_MODEL}/v2", BRIGHTNESS_MODEL],
    },
    "metrics": {
        "software_ssr": "exact payload matches / decoder-scenario tests",
        "original_ssr": "exact payload matches on original image only",
        "physical_ssr": "not measured automatically",
    },
    "rows": rows,
    "aggregates": aggregates,
}
(RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
with (RUN_DIR / "physical-ssr.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)
    writer.writerow(
        [
            "prompt_id",
            "method",
            "variant",
            "device",
            "medium",
            "attempt",
            "success",
            "latency_s",
            "notes",
        ]
    )
    for case in PROMPTS:
        for method, variant in keys:
            for device in ["Pixel 7", "iPhone 13"]:
                for attempt in range(1, 11):
                    writer.writerow(
                        [case["id"], method, variant, device, "screen", attempt, "", "", ""]
                    )
shutil.copy2(
    "/workspace/notebooks/08_diffqrcoder_vs_qrbtf_four_prompts.ipynb",
    RUN_DIR / "08_diffqrcoder_vs_qrbtf_four_prompts.ipynb",
)
archive = Path(
    shutil.make_archive(str(RUN_DIR), "gztar", root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name)
)
print("Archive :", archive, "SHA256=", hashlib.sha256(archive.read_bytes()).hexdigest())
print(
    "Serveur: POD=$(kubectl get pod -n qr-core -l app=prooftag-qr-notebook -o jsonpath='{.items[0].metadata.name}')"
)
print(f'Serveur: kubectl cp -n qr-core "${{POD}}:{archive}" "$HOME/{archive.name}"')
print(f'Windows: scp paul@pcIA:~/{archive.name} "$HOME/Downloads/"')

## Lecture correcte

- Un bon CLIP-aesthetic ne compense jamais un SSR faible.
- Le SSR logiciel robuste n'est pas le SSR physique du papier ; le CSV téléphone contient 10 tentatives par image et appareil.
- Le backend QRBTF exact reste une boîte noire : aucun nom de modèle ou latent intermédiaire n'est inventé.
- La comparaison locale contrôle la fondation en utilisant Cetus-Mix dans les deux branches. Elle compare donc les workflows publics, pas deux services commerciaux.
- Si SR-MPGD améliore le SSR mais détruit CLIP-aesthetic, le nombre d'itérations et le taux seront la prochaine ablation ; ils ne seront pas optimisés sur un seul prompt.